# KMeans full-dataset centroids — L2 only (v4, ERICA-only)

Reproduces the **L2-only** ERICA statistic table for a 4-component Gaussian mixture using the `erica` package.

Run target for this notebook version:
- 4-component Gaussian mixture
- `n (p) = 1000`, `dim = 100`
- normal variance (`variance_setting='base'`, diagonal covariance)
- K-means with `K = 2..6`, `B=200`, `P=80%`, `seed=123`


## Section 0 — Imports and seed

In [1]:
import os
import random
import numpy as np
import pandas as pd

from scipy.linalg import toeplitz

from erica.core import ERICA

seed = 123
np.random.seed(seed)
random.seed(seed)


## Section 1 — Parameters

In [2]:
# Table B.7 row 3 config: t = 100 (samples), p = 1000 (dims), Toeplitz, base variance.
dim              = 1000
n                = 100
variance_setting = 'base'      # 'low' | 'base' | 'high'
covariance_type  = 'toeplitz'  # 'diagonal' | 'toeplitz'
n_clusters_truth = 4

k_range          = [2, 3, 4, 5, 6]
B                = 200
percent_train    = 0.8

_cwd_abs = os.path.abspath(os.getcwd())
_experiments_root = (
    _cwd_abs if os.path.basename(_cwd_abs) == 'Experiments'
    else os.path.join(_cwd_abs, 'Experiments')
)
save_folder = os.path.join(
    _experiments_root,
    f'KMeansV4_L2_only_GaussMix{n_clusters_truth}'
    f'_n{n}_p{dim}_seed{seed}',
)
os.makedirs(save_folder, exist_ok=True)
print(f'save_folder    = {save_folder}')
print(f'k_range        = {k_range}')
print(f'B              = {B}')
print(f'percent_train  = {percent_train}')


save_folder    = /Users/shawnshirazi/LocalExperiments/ERICA/Experiments/KMeansV4_L2_only_GaussMix4_n100_p1000_seed123
k_range        = [2, 3, 4, 5, 6]
B              = 200
percent_train  = 0.8


## Section 2 — Synthetic data

Same 4-component Gaussian-mixture generator we've been using. Each component is $\mathcal{N}(c \cdot \mathbf{1}, 10 \cdot I_p)$ with $c \in \{1, 4, 7, 10\}$ and equal mixing weights. Because the components share a single isotropic covariance, whitening should *not* change the cluster geometry in any meaningful way — it just rescales and rotates axes.

In [3]:
mean_vectors = [c * np.ones(dim) for c in (1, 4, 7, 10)]
weights      = [0.25] * 4

variance_scale = {'low': 0.1, 'base': 1, 'high': 10}[variance_setting]

def create_covariance_matrix(dim, variance_scale, covariance_type):
    if covariance_type == 'diagonal':
        return variance_scale * np.identity(dim)
    if covariance_type == 'toeplitz':
        base = [1, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.001]
        col  = base + [0.0] * (dim - len(base))
        T    = toeplitz(col, col)
        T    = np.diag(np.diag(T) * variance_scale) + T - np.diag(np.diag(T))
        return T
    raise ValueError(f'Unknown covariance_type: {covariance_type}')

covariance_matrices = [
    create_covariance_matrix(dim, variance_scale, covariance_type)
    for _ in range(n_clusters_truth)
]

def generate_samples(n, dim, mean_vectors, covariance_matrices, weights):
    samples = np.empty((n, dim))
    labels  = np.empty((n, 1), dtype=int)
    for i in range(n):
        c = random.choices(range(len(weights)), weights=weights)[0]
        samples[i] = np.random.multivariate_normal(
            mean_vectors[c], covariance_matrices[c]
        )
        labels[i] = c
    return samples, labels

samples, component_list = generate_samples(
    n, dim, mean_vectors, covariance_matrices, weights
)

pd.DataFrame(samples).to_csv(
    os.path.join(save_folder, 'samples.csv'), index=False, header=False
)
pd.DataFrame(component_list).to_csv(
    os.path.join(save_folder, 'ground_truth_labels.csv'),
    index=False, header=False,
)

for c in range(n_clusters_truth):
    frac = float((component_list == c).sum()) / n
    print(f'  component {c}: fraction = {frac:.4f}')
print(f'samples shape: {samples.shape}')


  component 0: fraction = 0.3100
  component 1: fraction = 0.3000
  component 2: fraction = 0.1900
  component 3: fraction = 0.2000
samples shape: (100, 1000)


## Section 3 — Helper: run ERICA and tabulate

Runs the full `ERICA` orchestrator and returns a per-`k` metrics table (`CRI`, `WCRI`, `TWCRI`) with `K*` from `erica.k_star_`.


In [4]:
def run_erica_and_tabulate(data, output_subdir, label):
    print(f'=== Running ERICA pipeline: {label} ===')
    # reseed_rng=False preserves the post-data-generation RNG state so MCSS
    # splits match the canonical paper pipeline (matches mcss_kmeans2.py).
    # kmeans_n_init='auto' matches mcss_kmeans2.py literally; in sklearn>=1.4
    # this resolves to n_init=1 for k-means++ init. The Table B.7 row 1
    # comparison showed 'auto' tracks the paper's CRI more closely than legacy
    # n_init=10 (K=6 hits 0.733 vs paper 0.732).
    erica = ERICA(
        data          = data,
        k_range       = k_range,
        n_iterations  = B,
        train_percent = percent_train,
        method        = 'kmeans',
        random_seed   = seed,
        output_dir    = os.path.join(save_folder, output_subdir),
        transpose     = False,
        verbose       = True,
        reseed_rng    = False,
        kmeans_n_init = 'auto',
    )
    erica.run()

    rows = []
    for k in k_range:
        m = erica.metrics_.get(k, {}).get('kmeans', {})
        rows.append({
            'k':     k,
            'CRI':   m.get('CRI'),
            'WCRI':  m.get('WCRI'),
            'TWCRI': m.get('TWCRI'),
        })
    df = pd.DataFrame(rows).set_index('k')

    k_star = {
        'CRI':   erica.k_star_.get('CRI',   {}).get('kmeans'),
        'WCRI':  erica.k_star_.get('WCRI',  {}).get('kmeans'),
        'TWCRI': erica.k_star_.get('TWCRI', {}).get('kmeans'),
    }
    disq = dict(erica.disqualified_k_) if erica.disqualified_k_ else {}
    return df, k_star, disq, erica


## Section 4 — L2 pipeline (raw samples)

This is the only executed pipeline in v4.


In [5]:
l2_df, l2_kstar, l2_disq, l2_erica = run_erica_and_tabulate(
    samples, output_subdir='erica_run_l2', label='L2'
)

print()
for name, kv in l2_kstar.items():
    print(f'L2 K* ({name:5s}) = {kv}')
if l2_disq:
    print(f'L2 disqualified k: {l2_disq}')

def bold_kstar(kstar_dict):
    def styler(df):
        style = pd.DataFrame('', index=df.index, columns=df.columns)
        for col, kv in kstar_dict.items():
            if kv is not None and kv in style.index and col in style.columns:
                style.loc[kv, col] = 'font-weight: bold'
        return style
    return styler

(
    l2_df.style
        .format('{:.6f}')
        .apply(bold_kstar(l2_kstar), axis=None)
        .set_caption(
            f'L2 ERICA statistic — Gaussian mixture, p={dim}, '
            f'K-means, B={B}, P={int(percent_train*100)}%, '
            f'seed={seed}.  Bold = K* per Algorithm 2.'
        )
)


=== Running ERICA pipeline: L2 ===
ERICA initialized:
  Data shape: (100, 1000)
  K range: [2, 3, 4, 5, 6]
  Iterations: 200
  Methods: ['kmeans']
  Random seed: 123

Starting ERICA analysis at 2026-05-18 20:49:35

[1/5] Performing iterative clustering subsampling...
Starting iterative clustering subsampling...
  Subsampling iteration 1/200
  Subsampling iteration 51/200
  Subsampling iteration 101/200
  Subsampling iteration 151/200
Completed iterative clustering subsampling

[2/5] Running K-based clustering...
  Running K-Means clustering for k=2...
Starting K-Means clustering for k=2...
  Calculating global centroids...
  Running 200 iterations...
    Iteration 1/200
    Iteration 51/200
    Iteration 101/200
    Iteration 151/200
  Aligning cluster identities...
  Generating CLAM matrix...
  K-Means clustering complete for k=2
  Running K-Means clustering for k=3...
Starting K-Means clustering for k=3...
  Calculating global centroids...
  Running 200 iterations...
    Iteration 1/

,CRI,WCRI,TWCRI
k,,,
2,0.997390,0.498408,0.996816
3,0.927099,0.306759,0.920277
4,0.986468,0.246807,0.987229
5,nan,nan,nan
6,nan,nan,nan


## Section 4b — Paper-style ERICA statistic (CRI) table (Table B.7 format)

The "ERICA statistic" reported in Table B.7 of the paper **is** the CRI metric.
Per cluster, the CRI is the mean over samples primarily assigned to that cluster
of `max_count / total_count` across the `B` MCSS iterations. The headline
"ERICA statistic" per `K` is the mean of those per-cluster CRIs aggregated over
**non-empty** clusters only.

- **ERICA statistic (CRI)** = mean of per-cluster CRIs over non-empty clusters
  (empty clusters contribute `NA`, not 0). This is why `K=5,6` are still shown
  as finite values in the paper even though some clusters collapsed.
- **Per-cluster CRI** is shown in parentheses, with `NA` for empty clusters.
- **K\*** (bold) is selected by Algorithm 2 on the package-level metrics, which
  treats any `K` with an empty cluster as disqualified.

In [6]:
from erica.metrics import compute_cri

def paper_style_row(clam_matrix, k):
    """Build a (overall_stat, per_cluster_list) entry in the Table B.7 style.

    - overall_stat: mean of CRI over non-empty clusters (NaN-free aggregation)
    - per_cluster_list: list of length k, with float('nan') for empty clusters
    """
    cri = compute_cri(clam_matrix, k)
    primary = np.argmax(clam_matrix, axis=1)
    sizes = np.array([(primary == i).sum() for i in range(k)])
    per_cluster = [float('nan') if sizes[i] == 0 else float(cri[i]) for i in range(k)]
    nonempty = [v for v in per_cluster if not (isinstance(v, float) and np.isnan(v))]
    overall = float(np.mean(nonempty)) if nonempty else float('nan')
    return overall, per_cluster, sizes.tolist()


def fmt_per_cluster(per_cluster):
    parts = ['NA' if (isinstance(v, float) and np.isnan(v)) else f'{v:.3f}' for v in per_cluster]
    return '(' + ', '.join(parts) + ')'


paper_rows = []
for k in k_range:
    clam = l2_erica.clam_matrices_[(k, 'kmeans')]
    overall, per_cluster, sizes = paper_style_row(clam, k)
    paper_rows.append({
        'k': k,
        # The "ERICA statistic" reported in Table B.7 IS the CRI: it is the
        # mean of per-cluster CRIs aggregated over non-empty clusters.
        'ERICA statistic (CRI)': overall,
        'per-cluster CRI': fmt_per_cluster(per_cluster),
        'cluster sizes': sizes,
    })

paper_df = pd.DataFrame(paper_rows).set_index('k')

# K* is unchanged: Algorithm 2 disqualifies any K with an empty cluster
kstar_cri = l2_kstar.get('CRI')


def bold_paper_kstar(df):
    style = pd.DataFrame('', index=df.index, columns=df.columns)
    if kstar_cri in style.index:
        style.loc[kstar_cri, 'ERICA statistic (CRI)'] = 'font-weight: bold'
    return style


print('\n=== Paper-style ERICA statistic table (Table B.7 format) ===')
print('(The "ERICA statistic" column IS the CRI metric.)')
print(paper_df.to_string())
print(f"\nK* (CRI, Algorithm 2) = {kstar_cri}")
if l2_disq:
    print(f'Disqualified k (empty cluster): {l2_disq}')

(
    paper_df.style
        .format({'ERICA statistic (CRI)': lambda v: 'NaN' if pd.isna(v) else f'{v:.3f}'})
        .apply(bold_paper_kstar, axis=None)
        .set_caption(
            f'L2 ERICA statistic (CRI) — Gaussian mixture, '
            f'n={n}, p={dim}, K-means, B={B}, P={int(percent_train*100)}%, '
            f'variance={variance_setting}, seed={seed}. '
            f'Bold = K* per Algorithm 2.'
        )
)


=== Paper-style ERICA statistic table (Table B.7 format) ===
(The "ERICA statistic" column IS the CRI metric.)
   ERICA statistic (CRI)                       per-cluster CRI           cluster sizes
k                                                                                     
2               0.997390                        (0.995, 1.000)                [61, 39]
3               0.927099                 (0.953, 0.980, 0.848)            [31, 30, 39]
4               0.986468          (0.988, 0.991, 0.974, 0.992)        [31, 30, 19, 20]
5               0.826666      (0.751, 0.790, 0.870, 0.895, NA)     [31, 30, 19, 20, 0]
6               0.720900  (NA, 0.637, 0.631, 0.820, 0.797, NA)  [0, 31, 30, 19, 20, 0]

K* (CRI, Algorithm 2) = 4
Disqualified k (empty cluster): {'kmeans': [5, 6]}


,ERICA statistic (CRI),per-cluster CRI,cluster sizes
k,,,
2,0.997,"(0.995, 1.000)","[61, 39]"
3,0.927,"(0.953, 0.980, 0.848)","[31, 30, 39]"
4,0.986,"(0.988, 0.991, 0.974, 0.992)","[31, 30, 19, 20]"
5,0.827,"(0.751, 0.790, 0.870, 0.895, NA)","[31, 30, 19, 20, 0]"
6,0.721,"(NA, 0.637, 0.631, 0.820, 0.797, NA)","[0, 31, 30, 19, 20, 0]"


## Section 5 — Save L2 artifacts


In [7]:
l2_csv = os.path.join(save_folder, 'erica_statistic_table_l2.csv')
l2_df.to_csv(l2_csv)

md_path = os.path.join(save_folder, 'erica_statistic_table_l2.md')
with open(md_path, 'w') as f:
    f.write(
        f'# ERICA statistic — L2 only  \n'
        f'Gaussian mixture, p={n}, dim={dim}, K-means, B={B}, '
        f'P={int(percent_train*100)}%, seed={seed}, variance={variance_setting}\n\n'
    )
    f.write('| k | L2 CRI | L2 WCRI | L2 TWCRI |\n')
    f.write('|---|--------|---------|----------|\n')
    for k in k_range:
        def fmt(dist_df, kstar, col):
            v = dist_df.loc[k, col]
            s = f'{v:.6f}' if pd.notna(v) else 'NaN'
            return f'**{s}**' if kstar.get(col) == k else s
        f.write(
            f'| {k} '
            f'| {fmt(l2_df, l2_kstar, "CRI")} '
            f'| {fmt(l2_df, l2_kstar, "WCRI")} '
            f'| {fmt(l2_df, l2_kstar, "TWCRI")} |\n'
        )

kstar_path = os.path.join(save_folder, 'k_star_l2.txt')
with open(kstar_path, 'w') as f:
    for name in ('CRI', 'WCRI', 'TWCRI'):
        f.write(f'L2 K* ({name}) = {l2_kstar.get(name)}\n')
    f.write(f'L2 disqualified k: {l2_disq}\n')

paper_csv = os.path.join(save_folder, 'erica_statistic_table_l2_paper_style.csv')
paper_df.to_csv(paper_csv)

paper_md = os.path.join(save_folder, 'erica_statistic_table_l2_paper_style.md')
with open(paper_md, 'w') as f:
    f.write(
        f'# ERICA statistic — L2 (paper-style, Table B.7 format)  \n'
        f'Gaussian mixture, n={n}, p={dim}, K-means, B={B}, '
        f'P={int(percent_train*100)}%, seed={seed}, variance={variance_setting}\n\n'
    )
    f.write('| k | ERICA statistic (CRI) | Per-cluster CRI | Cluster sizes |\n')
    f.write('|---|-----------------------|------------------|----------------|\n')
    for k in k_range:
        v = paper_df.loc[k, 'ERICA statistic (CRI)']
        stat_s = 'NaN' if pd.isna(v) else f'{v:.3f}'
        if l2_kstar.get('CRI') == k:
            stat_s = f'**{stat_s}**'
        f.write(
            f"| {k} "
            f"| {stat_s} "
            f"| {paper_df.loc[k, 'per-cluster CRI']} "
            f"| {paper_df.loc[k, 'cluster sizes']} |\n"
        )

print(f'Saved L2 table      : {l2_csv}')
print(f'Saved markdown table: {md_path}')
print(f'Saved K* annotations: {kstar_path}')
print(f'Saved paper-style CSV : {paper_csv}')
print(f'Saved paper-style MD  : {paper_md}')
print(f'L2 ERICA run dir    : {l2_erica.output_folders_[-1]}')
display(l2_df)
display(paper_df)


Saved L2 table      : /Users/shawnshirazi/LocalExperiments/ERICA/Experiments/KMeansV4_L2_only_GaussMix4_n100_p1000_seed123/erica_statistic_table_l2.csv
Saved markdown table: /Users/shawnshirazi/LocalExperiments/ERICA/Experiments/KMeansV4_L2_only_GaussMix4_n100_p1000_seed123/erica_statistic_table_l2.md
Saved K* annotations: /Users/shawnshirazi/LocalExperiments/ERICA/Experiments/KMeansV4_L2_only_GaussMix4_n100_p1000_seed123/k_star_l2.txt
Saved paper-style CSV : /Users/shawnshirazi/LocalExperiments/ERICA/Experiments/KMeansV4_L2_only_GaussMix4_n100_p1000_seed123/erica_statistic_table_l2_paper_style.csv
Saved paper-style MD  : /Users/shawnshirazi/LocalExperiments/ERICA/Experiments/KMeansV4_L2_only_GaussMix4_n100_p1000_seed123/erica_statistic_table_l2_paper_style.md
L2 ERICA run dir    : /Users/shawnshirazi/LocalExperiments/ERICA/Experiments/KMeansV4_L2_only_GaussMix4_n100_p1000_seed123/erica_run_l2/erica_run_20260518_204935


,CRI,WCRI,TWCRI
k,,,
2,0.997390,0.498408,0.996816
3,0.927099,0.306759,0.920277
4,0.986468,0.246807,0.987229
5,NaN,NaN,NaN
6,NaN,NaN,NaN


,ERICA statistic (CRI),per-cluster CRI,cluster sizes
k,,,
2,0.997390,"(0.995, 1.000)","[61, 39]"
3,0.927099,"(0.953, 0.980, 0.848)","[31, 30, 39]"
4,0.986468,"(0.988, 0.991, 0.974, 0.992)","[31, 30, 19, 20]"
5,0.826666,"(0.751, 0.790, 0.870, 0.895, NA)","[31, 30, 19, 20, 0]"
6,0.720900,"(NA, 0.637, 0.631, 0.820, 0.797, NA)","[0, 31, 30, 19, 20, 0]"
